# Notebook 6: Graph Classification & Graph Isomorphism Networks (GIN)

All previous notebooks did *node* classification — predicting labels for individual nodes. Now we tackle **graph classification**: predict a label for an *entire graph*. Is this molecule mutagenic? Does this social network belong to a bot network?

**Prerequisites:** Notebooks 1–5

**What you'll learn:**
- Graph-level tasks and the pooling problem
- Global pooling operations (mean, sum, max)
- The Weisfeiler-Lehman (WL) graph isomorphism test
- GIN: theoretically the most powerful GNN class
- Training on molecular datasets (MUTAG, PROTEINS)
- Batch handling for multiple graphs with PyG's DataLoader

**By the end:** classify molecules as mutagenic/non-mutagenic with GIN.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch_geometric.datasets import TUDataset
from torch_geometric.nn import GINConv, GCNConv, global_mean_pool, global_add_pool, global_max_pool
from torch_geometric.loader import DataLoader
import networkx as nx
from torch_geometric.utils import to_networkx

torch.manual_seed(42)
np.random.seed(42)

---
## 1. The Graph Classification Problem

In node classification, each graph has ONE set of inputs/outputs and predictions are made per node.

In **graph classification**, we have a *dataset of many graphs*, each with its own label:
- MUTAG: 188 small molecules, label = mutagenic or not
- PROTEINS: 1113 graphs (proteins), label = enzyme or not
- COLLAB: 5000 scientific collaboration graphs

### The Batching Challenge

Each graph has a different number of nodes! We can't just stack them into a tensor. PyG's DataLoader solves this by creating a **block-diagonal adjacency matrix** — merging all graphs in a batch into one big disconnected graph:

```
Batch of 3 graphs:
Graph 1: 5 nodes, edges within nodes 0-4
Graph 2: 3 nodes, edges within nodes 5-7 (renumbered)
Graph 3: 7 nodes, edges within nodes 8-14
-> One big graph: 15 nodes, no edges across graph boundaries
```

A `batch` vector tracks which graph each node belongs to.

In [ ]:
# Load MUTAG dataset
dataset = TUDataset(root='/tmp/MUTAG', name='MUTAG')
print(f'Dataset: {dataset}')
print(f'Num graphs: {len(dataset)}')
print(f'Num classes: {dataset.num_classes}')
print(f'Num node features: {dataset.num_node_features}')
print()

# Inspect a few graphs
for i in range(3):
    d = dataset[i]
    print(f'Graph {i}: {d.num_nodes} nodes, {d.num_edges} edges, label={d.y.item()}')

print()

# Label distribution
labels = [d.y.item() for d in dataset]
from collections import Counter
print('Label distribution:', Counter(labels))
print('  Label 1 = mutagenic, Label 0 = non-mutagenic')

# Node count distribution
node_counts = [d.num_nodes for d in dataset]
print(f'\nNode counts: min={min(node_counts)}, max={max(node_counts)}, '
      f'mean={np.mean(node_counts):.1f}')

In [ ]:
# Visualize 6 MUTAG molecules
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
class_names = ['non-mutagenic', 'mutagenic']
colors = ['steelblue', 'coral']

for idx, ax in enumerate(axes.flatten()):
    d = dataset[idx]
    g = to_networkx(d, to_undirected=True)
    label = d.y.item()
    
    # Node atom type (feature is one-hot, take argmax)
    atom_types = d.x.argmax(dim=1).tolist() if d.x is not None else [0]*d.num_nodes
    atom_colors = plt.cm.Set2(np.array(atom_types) / max(max(atom_types), 1))
    
    pos = nx.spring_layout(g, seed=idx)
    nx.draw(g, pos, ax=ax, node_color=atom_colors, node_size=400,
            edge_color='gray', with_labels=True, font_size=7)
    ax.set_title(f'Graph {idx} — {class_names[label]}', fontsize=10,
                color=colors[label])

plt.suptitle('MUTAG Molecules', fontsize=14)
plt.tight_layout()
plt.show()

---
## 2. Global Pooling: From Node Embeddings to Graph Embeddings

After message passing, each node has an embedding. To classify the *graph*, we need a single vector. **Global pooling** aggregates all node embeddings:

| Pooling | Formula | Properties |
|---------|---------|----------|
| **Mean** | $\frac{1}{N}\sum_i h_i$ | Normalized, good for variable-size graphs |
| **Sum (Add)** | $\sum_i h_i$ | Sensitive to graph size, GIN uses this |
| **Max** | $\max_i h_i$ | Captures dominant features |

**Key insight:** GIN uses **sum** pooling because it's injective — different multisets of node features give different sums. Mean pooling can confuse graphs of different sizes.

In [ ]:
# Illustrate the difference between pooling methods
# Two different graphs that MEAN pooling can't distinguish, but SUM can

hidden = 4  # embedding dimension

# Graph A: 2 nodes with embedding [1, 0, 0, 0]
# Graph B: 4 nodes with embedding [1, 0, 0, 0]
embedding_a = torch.tensor([[1., 0., 0., 0.], [1., 0., 0., 0.]])
embedding_b = torch.tensor([[1., 0., 0., 0.], [1., 0., 0., 0.],
                             [1., 0., 0., 0.], [1., 0., 0., 0.]])

batch_a = torch.zeros(2, dtype=torch.long)  # both nodes belong to graph 0
batch_b = torch.zeros(4, dtype=torch.long)  # all nodes belong to graph 0

mean_a = global_mean_pool(embedding_a, batch_a)
mean_b = global_mean_pool(embedding_b, batch_b)
sum_a  = global_add_pool(embedding_a, batch_a)
sum_b  = global_add_pool(embedding_b, batch_b)

print('Two different graphs with identical node features:')
print(f'Graph A (2 nodes): mean={mean_a.tolist()}, sum={sum_a.tolist()}')
print(f'Graph B (4 nodes): mean={mean_b.tolist()}, sum={sum_b.tolist()}')
print()
print('Mean pooling cannot distinguish them!')
print('Sum pooling CAN distinguish them — this is why GIN uses sum.')

---
## 3. The Weisfeiler-Lehman Test & GIN

**The WL test** is a classical algorithm for graph isomorphism (are two graphs structurally identical?):
1. Initialize all nodes with the same label
2. Iteratively: update each node's label = hash(own_label, multiset of neighbor labels)
3. If two graphs get the same set of labels → potentially isomorphic

**Key result (Xu et al., 2019):** Any GNN using sum aggregation + injective update function is AS POWERFUL as the WL test — the theoretical maximum for message-passing GNNs.

**GIN achieves this** with:
$$h_v^{(k)} = \text{MLP}^{(k)}\left((1 + \epsilon^{(k)}) \cdot h_v^{(k-1)} + \sum_{u \in \mathcal{N}(v)} h_u^{(k-1)}\right)$$

- Sum aggregation (not mean)
- MLP update (more expressive than a single linear layer)
- $\epsilon$ is learnable (or fixed at 0)

In [ ]:
# ============================================================
# GIN from scratch
# ============================================================
from torch_geometric.nn import MessagePassing

class GINConvScratch(MessagePassing):
    """
    GIN layer from scratch.
    h_v^new = MLP((1 + eps) * h_v + sum(h_u for u in N(v)))
    """
    def __init__(self, in_channels, out_channels, eps=0.0, train_eps=False):
        super().__init__(aggr='add')  # sum aggregation — critical for WL expressiveness
        
        # MLP: two linear layers with batch norm
        self.mlp = nn.Sequential(
            nn.Linear(in_channels, out_channels),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(),
            nn.Linear(out_channels, out_channels),
        )
        # epsilon: learnable or fixed
        if train_eps:
            self.eps = nn.Parameter(torch.tensor([eps]))
        else:
            self.register_buffer('eps', torch.tensor([eps]))
    
    def forward(self, x, edge_index):
        # Aggregate neighbors with sum
        neighbor_agg = self.propagate(edge_index, x=x)
        # Combine: (1 + eps) * own + neighbor_sum
        out = (1 + self.eps) * x + neighbor_agg
        return self.mlp(out)
    
    def message(self, x_j):
        return x_j  # just pass neighbor features, aggr='add' sums them


# Test
gin_layer = GINConvScratch(dataset.num_node_features, 32, train_eps=True)
dummy_data = dataset[0]
out = gin_layer(dummy_data.x.float(), dummy_data.edge_index)
print(f'GIN scratch output: {out.shape}')

In [ ]:
# ============================================================
# Full GIN Graph Classifier using PyG's GINConv
# ============================================================

class GINGraphClassifier(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels,
                 num_layers=5, dropout=0.5):
        super().__init__()
        self.dropout = dropout
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        for i in range(num_layers):
            in_ch = in_channels if i == 0 else hidden_channels
            mlp = nn.Sequential(
                nn.Linear(in_ch, hidden_channels),
                nn.BatchNorm1d(hidden_channels),
                nn.ReLU(),
                nn.Linear(hidden_channels, hidden_channels),
            )
            self.convs.append(GINConv(mlp, train_eps=True))
            self.bns.append(nn.BatchNorm1d(hidden_channels))
        
        # Classifier head: pooled embedding -> class
        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels * num_layers, hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, out_channels)
        )
    
    def forward(self, x, edge_index, batch):
        # Collect embeddings from ALL layers (sum-pool each, then concat)
        pooled_per_layer = []
        
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            pooled = global_add_pool(x, batch)  # (batch_size, hidden_channels)
            pooled_per_layer.append(pooled)
        
        # Concatenate pooled representations from all layers
        h_graph = torch.cat(pooled_per_layer, dim=-1)  # (batch_size, hidden*num_layers)
        return self.classifier(h_graph)


torch.manual_seed(42)
model_gin = GINGraphClassifier(
    in_channels=dataset.num_node_features,
    hidden_channels=64,
    out_channels=dataset.num_classes,
    num_layers=5,
    dropout=0.5
)
print(model_gin)
print(f'\nTotal params: {sum(p.numel() for p in model_gin.parameters()):,}')

In [ ]:
# ============================================================
# Data preparation: train/val/test split + DataLoader
# ============================================================

# Shuffle dataset
torch.manual_seed(42)
perm = torch.randperm(len(dataset))
dataset = dataset[perm]

# 10-fold split (standard for MUTAG)
n = len(dataset)
train_end = int(0.7 * n)
val_end = int(0.85 * n)

train_dataset = dataset[:train_end]
val_dataset = dataset[train_end:val_end]
test_dataset = dataset[val_end:]

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32)
test_loader  = DataLoader(test_dataset,  batch_size=32)

# Inspect a batch
batch = next(iter(train_loader))
print(f'\nBatch: {batch}')
print(f'batch.batch: {batch.batch[:20]}...')  # which graph each node belongs to
print(f'Unique graphs in batch: {batch.batch.max().item() + 1}')

In [ ]:
def train_graph_clf(model, loader, optimizer):
    model.train()
    total_loss = 0
    for batch in loader:
        optimizer.zero_grad()
        out = model(batch.x.float(), batch.edge_index, batch.batch)
        loss = F.cross_entropy(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)


@torch.no_grad()
def eval_graph_clf(model, loader):
    model.eval()
    correct = 0
    for batch in loader:
        out = model(batch.x.float(), batch.edge_index, batch.batch)
        pred = out.argmax(dim=1)
        correct += (pred == batch.y).sum().item()
    return correct / len(loader.dataset)


torch.manual_seed(42)
model_gin = GINGraphClassifier(dataset.num_node_features, 64, dataset.num_classes, num_layers=5)
optimizer_gin = torch.optim.Adam(model_gin.parameters(), lr=0.01)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer_gin, step_size=50, gamma=0.5)

history = {'loss': [], 'train_acc': [], 'val_acc': [], 'test_acc': []}
best_val, best_test = 0, 0

for epoch in range(1, 201):
    loss = train_graph_clf(model_gin, train_loader, optimizer_gin)
    scheduler.step()
    train_acc = eval_graph_clf(model_gin, train_loader)
    val_acc   = eval_graph_clf(model_gin, val_loader)
    test_acc  = eval_graph_clf(model_gin, test_loader)
    
    history['loss'].append(loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['test_acc'].append(test_acc)
    
    if val_acc > best_val:
        best_val, best_test = val_acc, test_acc
    
    if epoch % 50 == 0:
        print(f'Epoch {epoch:>3} | Loss: {loss:.4f} | Train: {train_acc:.4f} | '
              f'Val: {val_acc:.4f} | Test: {test_acc:.4f}')

print(f'\nBest Val: {best_val:.4f} | Test at best val: {best_test:.4f}')

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history['loss'], color='red')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_title('Training Loss')

axes[1].plot(history['train_acc'], label='Train', color='steelblue')
axes[1].plot(history['val_acc'],   label='Val',   color='orange')
axes[1].plot(history['test_acc'],  label='Test',  color='green', linestyle='--')
axes[1].axhline(best_test, color='gray', linestyle=':', label=f'Best test: {best_test:.4f}')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy — Graph Classification (MUTAG)')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 4. GIN vs. GCN vs. GraphSAGE on Graph Classification

In [ ]:
# GCN-based graph classifier for comparison
class GCNGraphClassifier(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5):
        super().__init__()
        self.dropout = dropout
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.conv3 = GCNConv(hidden_channels, hidden_channels)
        self.lin = nn.Linear(hidden_channels, out_channels)
    
    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.lin(x)


def run_graph_clf(ModelClass, **kwargs):
    torch.manual_seed(42)
    m = ModelClass(**kwargs)
    opt = torch.optim.Adam(m.parameters(), lr=0.01)
    sch = torch.optim.lr_scheduler.StepLR(opt, step_size=50, gamma=0.5)
    best_v, best_t = 0, 0
    for epoch in range(200):
        train_graph_clf(m, train_loader, opt); sch.step()
        v = eval_graph_clf(m, val_loader)
        t = eval_graph_clf(m, test_loader)
        if v > best_v: best_v, best_t = v, t
    return best_v, best_t


print('Running comparisons (may take ~1-2 min)...')
gin_v, gin_t = run_graph_clf(
    GINGraphClassifier,
    in_channels=dataset.num_node_features, hidden_channels=64,
    out_channels=dataset.num_classes, num_layers=5
)
gcn_v, gcn_t = run_graph_clf(
    GCNGraphClassifier,
    in_channels=dataset.num_node_features, hidden_channels=64,
    out_channels=dataset.num_classes
)

print()
print('=== MUTAG Graph Classification ===')
print(f'{"Model":<15} | {"Val Acc":>8} | {"Test Acc":>9}')
print('-' * 38)
print(f'{"GIN (5-layer)":<15} | {gin_v:>8.4f} | {gin_t:>9.4f}')
print(f'{"GCN (3-layer)":<15} | {gcn_v:>8.4f} | {gcn_t:>9.4f}')

---
# MINI-PROJECT 6: Classify Proteins with GIN

**Task:** Apply GIN to the **PROTEINS** dataset — classify proteins as enzymes or non-enzymes. This is a harder, larger dataset (1113 graphs, 2 classes).

**Requirements:**
1. Load `TUDataset('PROTEINS')` and explore it (graph size distribution, class balance)
2. Implement a `GINGraphClassifier` — try different `num_layers` (3, 4, 5) and `hidden_channels` (32, 64, 128)
3. Use 10-fold cross-validation (split dataset into 10 folds, train on 9, test on 1, rotate)
4. Report mean ± std accuracy across folds
5. Compare GIN with mean pooling vs. sum pooling vs. max pooling
6. Plot: training curve for the best configuration

**Stretch goal:** Try `IMDB-BINARY` (no node features — you'll need to add degree as a feature).

In [ ]:
# MINI-PROJECT SKELETON

# TODO 1: Load PROTEINS
# dataset_prot = TUDataset(root='/tmp/PROTEINS', name='PROTEINS')
# Explore: num_graphs, num_classes, num_features, graph size stats


# TODO 2: 10-fold cross-validation
def ten_fold_cv(dataset, ModelClass, model_kwargs, lr=0.01, epochs=100):
    """
    Run 10-fold CV and return list of test accuracies.
    """
    n = len(dataset)
    fold_size = n // 10
    accs = []
    
    for fold in range(10):
        # Split into train / test
        test_idx = list(range(fold * fold_size, (fold + 1) * fold_size))
        train_idx = list(range(0, fold * fold_size)) + list(range((fold + 1) * fold_size, n))
        
        train_ds = dataset[train_idx]
        test_ds  = dataset[test_idx]
        
        train_loader_cv = DataLoader(train_ds, batch_size=32, shuffle=True)
        test_loader_cv  = DataLoader(test_ds,  batch_size=32)
        
        # TODO: instantiate model, train for `epochs` epochs, evaluate
        # acc = eval_graph_clf(model, test_loader_cv)
        # accs.append(acc)
        pass
    
    return accs


# TODO 3: Run CV for different configs
# configs = [
#     {'num_layers': 3, 'hidden_channels': 64},
#     {'num_layers': 5, 'hidden_channels': 64},
#     {'num_layers': 5, 'hidden_channels': 128},
# ]
# for cfg in configs:
#     accs = ten_fold_cv(dataset_prot, GINGraphClassifier, {**cfg, ...})
#     print(f'{cfg} -> {np.mean(accs):.4f} ± {np.std(accs):.4f}')


# TODO 4: Pooling comparison
# Modify GINGraphClassifier to accept a pooling argument (mean/add/max)
# Compare global_mean_pool vs global_add_pool vs global_max_pool

print('Implement the TODO sections above!')

### Hints

<details>
<summary>Hint 1 — IMDB-BINARY has no node features</summary>

For graphs without node features, use node degree as a feature:
```python
from torch_geometric.transforms import OneHotDegree
# Adds one-hot encoded degree as node feature
dataset = TUDataset(root='/tmp/IMDB', name='IMDB-BINARY',
                    transform=OneHotDegree(max_degree=135))
```
</details>

<details>
<summary>Hint 2 — Parameterizable pooling in GIN</summary>

```python
from torch_geometric.nn import global_mean_pool, global_add_pool, global_max_pool

pooling_fn = {'mean': global_mean_pool,
              'add': global_add_pool,
              'max': global_max_pool}[pooling]
```
Pass `pooling='mean'` (etc.) to the constructor.
</details>

---
## What's Next?

In **Notebook 7**, we bring everything together with **real OGB benchmarks**, **link prediction**, and a comparison of all models we've built. You'll also get a final capstone project.